In [1]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
from dotenv import load_dotenv
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

/raid/home/m13521157/absa-sft-comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [3]:
from typing import List, Dict, Any, Literal

In [4]:
def parse_absa_string(text: str):
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

def convert_to_absa_format(triplets: List[Dict[str, str]], order: List[Literal['A', 'O', 'S']]) -> str:
	"""
	Converts a list of dictionaries containing ABSA triplets into a formatted string.
	Each dictionary should contain keys 'A', 'O', and 'S' for Aspect, Opinion, and Sentiment respectively.
	The order of these elements in the output string is determined by the 'order' parameter.

	Args:
		triplets (List[Dict[str, str]]): List of dictionaries with ABSA triplet information.
	Returns:
		str: A formatted string representing the ABSA triplets.
	"""
	result = []
	for triplet in triplets:
		parts = []
		for key in order:
			if key in triplet:
				parts.append(f"[{key}] {triplet[key]}")
		result.append(" ".join(parts))
	return " [SSEP] ".join(result)

def convert_to_gas_format(data_list, spaced=False):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A, O, S); (A, O, S); ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	if spaced:
		triplets = [f"( {item['A']} | {item['O']} | {item['S']} )" for item in data_list]
	else:
		triplets = [f"({item['A']}| {item['O']}| {item['S']})" for item in data_list]

	# Join the list of strings together with a semicolon and space
	if spaced:
		return " ; ".join(triplets)
	return "; ".join(triplets)

## New Data

In [29]:
from copy import deepcopy

In [30]:
orders = [['A', 'O'], ['A', 'S'], ['A'], ['O']]

In [31]:
# Order to task
order2task = {
    ('A', 'O'): 'ao',
    ('A', 'S'): 'as',
	('A',): 'a',
	('O',): 'o'
}

In [32]:
for split in ['train', 'dev', 'test']:
	data_path = f'dataset/hoasa_hotel/indo/mvp_aos/{split}.json'
	with open(data_path, 'r') as f:
		data = json.load(f)

	# For each order, create a new dataset
	for order in orders:
		new_data = []
		for instance in data:
			parsed_data = parse_absa_string(instance['target'])
			# Remove all duplicate entries in parsed_data
			parsed_data = [dict(t) for t in {tuple(d.items()) for d in parsed_data}]
			new_parsed_data = []
			for triplet in parsed_data:
				new_triplet = {}
				for key in order:
					if key in triplet:
						new_triplet[key] = triplet[key]
				new_parsed_data.append(new_triplet)
			new_data.append({
				'sentence_id': instance['sentence_id'],
				'instance_id': instance['instance_id'],
				'input': instance['input'].replace('[A] [O] [S]', " ".join([f'[{key}]' for key in order])),
				'target': convert_to_absa_format(new_parsed_data, order=order),
				'element_order': ''.join(order).lower(),
				'task_elements': ''.join(order).lower(),
				'dataset_type': deepcopy(instance['dataset_type']),
			})
		
		# Save new_data to json
		save_path = f'dataset/hoasa_hotel_{order2task[tuple(order)]}/indo/mvp_aos/{split}.json'
		os.makedirs(os.path.dirname(save_path), exist_ok=True)
		with open(save_path, 'w') as f:
			json.dump(new_data, f, indent=4, ensure_ascii=False)
		print(f'Saved {save_path}')

Saved dataset/hoasa_hotel_ao/indo/mvp_aos/train.json
Saved dataset/hoasa_hotel_as/indo/mvp_aos/train.json
Saved dataset/hoasa_hotel_a/indo/mvp_aos/train.json
Saved dataset/hoasa_hotel_o/indo/mvp_aos/train.json
Saved dataset/hoasa_hotel_ao/indo/mvp_aos/dev.json
Saved dataset/hoasa_hotel_as/indo/mvp_aos/dev.json
Saved dataset/hoasa_hotel_a/indo/mvp_aos/dev.json
Saved dataset/hoasa_hotel_o/indo/mvp_aos/dev.json
Saved dataset/hoasa_hotel_ao/indo/mvp_aos/test.json
Saved dataset/hoasa_hotel_as/indo/mvp_aos/test.json
Saved dataset/hoasa_hotel_a/indo/mvp_aos/test.json
Saved dataset/hoasa_hotel_o/indo/mvp_aos/test.json
